In [ ]:
from pathlib import Path
import re
import xarray as xr
import pandas as pd

from bias_to_grid_comparison import _coerce_finite_lonlat
import geopandas as gpd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
from shapely.ops import unary_union
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from cartopy.mpl.path import shapely_to_path





def plot_merged_map_cartopy_nice(
    da: xr.DataArray,
    *,
    outpath: Path,
    title: str,
    center: float,
    onshore_geojson: Path,
    offshore_geojson: Path,
    points: pd.DataFrame,
    value_col: str,
    extent: tuple[float, float, float, float] | None = None,
    dpi: int = 300,
    projection: str = "laea",
    robust: bool = True,
    q: tuple[float, float] = (0.02, 0.98),
    aoi_buffer_deg: float = 0.35,
) -> None:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    import cartopy.crs as ccrs
    import cartopy.feature as cfeature
    from cartopy.mpl.path import shapely_to_path
    from shapely.ops import unary_union
    from mpl_toolkits.axes_grid1.inset_locator import inset_axes

    # ----------------- helpers -----------------
    def _extent_from_da_finite(da_, pad_frac: float = 0.01):
        lon_ = np.asarray(da_["lon"].values, dtype=float)
        lat_ = np.asarray(da_["lat"].values, dtype=float)
        z_ = np.asarray(da_.values, dtype=float)
        m = np.isfinite(z_)
        if not np.any(m):
            return None
        rows = np.where(m.any(axis=1))[0]
        cols = np.where(m.any(axis=0))[0]
        latmin = float(lat_[rows.min()]); latmax = float(lat_[rows.max()])
        lonmin = float(lon_[cols.min()]); lonmax = float(lon_[cols.max()])
        dx = (lonmax - lonmin) * pad_frac
        dy = (latmax - latmin) * pad_frac
        return (lonmin - dx, lonmax + dx, latmin - dy, latmax + dy)

    def _robust_vmin_vmax(z_, center_, q_=(0.02, 0.98)):
        zz = np.asarray(z_, dtype=float)
        zz = zz[np.isfinite(zz)]
        if zz.size == 0:
            return center_ - 1.0, center_ + 1.0
        lo = np.quantile(zz, q_[0])
        hi = np.quantile(zz, q_[1])
        # make symmetric around center for diverging colormap
        d = max(abs(lo - center_), abs(hi - center_))
        return center_ - d, center_ + d

    # ----------------- sanitize grid -----------------
    lon = da["lon"].values
    lat = da["lat"].values
    if isinstance(lon, np.ma.MaskedArray): lon = lon.filled(np.nan)
    if isinstance(lat, np.ma.MaskedArray): lat = lat.filled(np.nan)
    lon = np.asarray(lon, dtype=float)
    lat = np.asarray(lat, dtype=float)

    m_lon = np.isfinite(lon)
    m_lat = np.isfinite(lat)
    if not m_lon.all() or not m_lat.all():
        da = da.isel(lon=np.where(m_lon)[0], lat=np.where(m_lat)[0])
        lon = lon[m_lon]
        lat = lat[m_lat]

    z = da.values
    if isinstance(z, np.ma.MaskedArray): z = z.filled(np.nan)
    z = np.asarray(z, dtype=float)

    lon2, lat2 = np.meshgrid(lon, lat)

    # ----------------- bounds / colors -----------------
    if robust:
        vmin, vmax = _robust_vmin_vmax(z, center, q_=q)
    else:
        # symmetric full range
        zz = z[np.isfinite(z)]
        d = np.nanmax(np.abs(zz - center)) if zz.size else 1.0
        vmin, vmax = center - d, center + d

    cmap = "PRGn"  # keep your diverging palette

    # ----------------- polygons -----------------
    g_on = gpd.read_file(onshore_geojson)
    keep = ["GB", "FR", "DE", "NL", "BE", "DK", "NO"]

    g_on = g_on[g_on["name"].isin(keep)].copy()
    g_off = gpd.read_file(offshore_geojson)
    g_on = g_on.set_crs("EPSG:4326") if g_on.crs is None else g_on.to_crs("EPSG:4326")
    g_off = g_off.set_crs("EPSG:4326") if g_off.crs is None else g_off.to_crs("EPSG:4326")

    # AOI boundary clip
    aoi_union = unary_union(list(g_on.geometry) + list(g_off.geometry))
    aoi_union = aoi_union.buffer(aoi_buffer_deg).buffer(0)  # clean

    # ----------------- points -----------------
    p = _coerce_finite_lonlat(points, name="controls")
    if "mode" not in p.columns:
        p["mode"] = "onshore"

    pon = p.loc[p["mode"].astype(str).str.lower().eq("onshore")]
    poff = p.loc[p["mode"].astype(str).str.lower().eq("offshore")]

    # ----------------- projection -----------------
    if projection.lower() == "laea":
        proj = ccrs.LambertAzimuthalEqualArea(central_longitude=10, central_latitude=55)
    else:
        proj = ccrs.PlateCarree()
    data_crs = ccrs.PlateCarree()

    # extent
    if extent is None:
        extent = _extent_from_da_finite(da, pad_frac=0.01)
    if extent is None:
        extent = (-12, 16, 45, 62)

    # ----------------- plot -----------------
    fig = plt.figure(figsize=(7.2, 9.0))  # thesis-friendly portrait
    ax = plt.axes(projection=proj)
    ax.set_extent([extent[0], extent[1], extent[2], extent[3]], crs=data_crs)

    # background (subtle)
    # ax.add_feature(cfeature.OCEAN, zorder=0)
    # ax.add_feature(cfeature.LAND, zorder=0)
    # ax.coastlines(resolution="50m", linewidth=0.4)

    # # clip to AOI (removes projection corner whitespace)
    # ax.set_boundary(shapely_to_path(aoi_union), transform=data_crs)

    # raster
    im = ax.pcolormesh(
        lon2, lat2, z,
        shading="auto",
        vmin=vmin, vmax=vmax,
        cmap=cmap,
        rasterized=True,
        transform=data_crs,
        zorder=1,
    )

    # Optional AOI outlines (keep VERY subtle, or turn off)
    DRAW_AOI = True  # <-- set True only if you really want outlines

    if DRAW_AOI:
        ax.add_geometries(g_on.geometry, crs=data_crs, facecolor="none",
                        edgecolor="black", linewidth=0.25, alpha=0.25, zorder=3)
        ax.add_geometries(g_off.geometry, crs=data_crs, facecolor="none",
                        edgecolor="black", linewidth=0.25, alpha=0.25, zorder=3)

    max_plot_points = 2500
    if len(pon) > max_plot_points:
        pon = pon.sample(max_plot_points, random_state=0)

    max_plot_points_off = 800
    if len(poff) > max_plot_points_off:
        poff = poff.sample(max_plot_points_off, random_state=0)
        
    # points (neutral so surface is the focus)
    if len(pon) > 0:
        ax.scatter(
            pon["lon"], pon["lat"],
            s=8, facecolor="none", edgecolor="k",
            linewidth=0.2, alpha=0.35,
            marker="o", label="Onshore",
            transform=data_crs, zorder=4
        )
    if len(poff) > 0:
        ax.scatter(
            poff["lon"], poff["lat"],
            s=8, facecolor="none", edgecolor="k",
            linewidth=0.2, alpha=0.35,
            marker="^", label="Offshore",
            transform=data_crs, zorder=4
        )

    # inset colorbar (doesn't shrink map)
    cax = inset_axes(ax, width="3.2%", height="45%", loc="center right", borderpad=1)
    cb = fig.colorbar(im, cax=cax)
    cb.set_label(value_col.capitalize())

    ax.set_title(title)
    ax.legend(loc="lower left", frameon=True, title="Control points", fontsize=8)

    outpath = Path(outpath)
    outpath.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(outpath, dpi=int(dpi), bbox_inches="tight", pad_inches=0.02, facecolor="white")
    plt.close(fig)

def plot_surfaces_from_nc(
    *,
    surfaces_dir: Path,
    plots_dir: Path,
    onshore_geojson: Path,
    offshore_geojson: Path,
    controls_csv: Path | None = None,
    controls_df: pd.DataFrame | None = None,
    dpi: int = 250,
) -> Path:
    """
    Create merged maps by reading precomputed surface NetCDF files
    (e.g. out_dir/surfaces/scalar_merged_ok.nc) without recomputing interpolations.

    Expects files named: {value_col}_merged_{method}.nc
      - value_col in {"scalar","offset"}
      - method like {"nearest","idw","ok","rbf",...}

    The NetCDF can contain either:
      - a single DataArray variable, or
      - a Dataset with one main variable; we will select the first data var.

    Controls (points) can be passed via controls_csv or controls_df.
    """
    surfaces_dir = Path(surfaces_dir)
    plots_dir = Path(plots_dir)
    plots_dir.mkdir(parents=True, exist_ok=True)

    # Load controls once (optional; only for point overlay)
    controls = None
    if controls_df is not None:
        controls = controls_df.copy()
    elif controls_csv is not None:
        controls = pd.read_csv(controls_csv)

    # match your naming convention
    pat = re.compile(r"^(scalar|offset)_merged_(.+)\.nc$")

    nc_files = sorted(surfaces_dir.glob("*.nc"))
    if not nc_files:
        raise FileNotFoundError(f"No .nc files found in {surfaces_dir}")

    for nc in nc_files:
        m = pat.match(nc.name)
        if not m:
            # skip unrelated nc files
            continue

        value_col = m.group(1)
        method = m.group(2)

        center = 1.0 if value_col == "scalar" else 0.0

        # Load surface
        ds = xr.open_dataset(nc)

        # Pick the main variable robustly
        if value_col in ds.data_vars:
            da = ds[value_col]
        else:
            # fall back to first data var
            da = ds[list(ds.data_vars)[0]]

        # Ensure lon/lat coords exist (plot_merged_map expects them)
        # (If you later switch to x/y, we can adapt plot_merged_map too.)
        if "lon" not in da.coords or "lat" not in da.coords:
            raise KeyError(
                f"{nc} does not have lon/lat coords. Found coords={list(da.coords)}. "
                "Either export lon/lat coords, or update plot_merged_map to use x/y."
            )

        out_png = plots_dir / f"{value_col}_merged_{method}.png"

        plot_merged_map_cartopy_nice(
            da,
            outpath=out_png,
            title=f"{method.upper()} — {value_col.title()} (Merged Onshore+Offshore)",
            center=center,
            onshore_geojson=onshore_geojson,
            offshore_geojson=offshore_geojson,
            points=controls,
            value_col=value_col,
            robust=True,
            q=(0.02, 0.98),
        )

    return plots_dir

In [53]:
out_dir = Path("out/bias_to_grid")  # <-- your run directory

plot_surfaces_from_nc(
    surfaces_dir=out_dir / "surfaces",
    plots_dir=out_dir / "test/maps",
    onshore_geojson=Path("input/regions/country_shapes.geojson"),
    offshore_geojson=Path("input/regions/north_sea_shape.geojson"),
    controls_csv=Path("out/correction_points.csv"),  # optional overlay
    dpi=250,
    # tell it which plotting function to use (see below)
)

PosixPath('out/bias_to_grid/test/maps')

In [47]:
gpd.read_file("input/regions/country_shapes.geojson")

,name,geometry
0,FR,"MULTIPOLYGON (((2.5218 51.08754, 2.53703 51.06..."
1,DE,"MULTIPOLYGON (((13.81572 48.76643, 13.78586 48..."
2,NO,"MULTIPOLYGON (((20.62316 69.03636, 20.36272 69..."
3,BE,"POLYGON ((2.5218 51.08754, 2.542 51.09687, 2.5..."
4,DK,"MULTIPOLYGON (((8.66078 54.89631, 8.66879 54.9..."
5,GB,"MULTIPOLYGON (((-2.66535 51.61725, -2.69376 51..."
6,NL,"MULTIPOLYGON (((7.19459 53.24502, 7.19747 53.2..."
